# Batch Silver Dim's

In [0]:
# =========================================
# 1. CREATE SCHEMA WITH EXTERNAL LOCATION
# =========================================
catalog = "eus-sales-catalog"
schema = "silver-layer"

external_location = "abfss://silver-layer@<storage-account>.dfs.core.windows.net/"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`
MANAGED LOCATION '{external_location}'
""")

print("✅ Schema ready")


# =========================================
# 2. IMPORTS
# =========================================
from pyspark.sql.functions import *
from pyspark.sql.types import MapType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable


# =========================================
# 3. HANDLE SCHEMA DRIFT
# =========================================
def handle_schema_drift(df):

    df = df.withColumn(
        "rescued_map",
        from_json(col("_rescued_data"), MapType(StringType(), StringType()))
    )

    rescued_keys = (
        df.selectExpr("explode(map_keys(rescued_map)) as key")
          .distinct()
          .selectExpr("collect_list(key) as keys")
          .collect()[0]["keys"]
    )

    ignore_keys = ["_file_path", "_corrupt_record"]
    schema_dict = {f.name.lower(): f.dataType for f in df.schema.fields}

    for key in rescued_keys:
        if key.lower() in ignore_keys:
            continue

        normalized_key = key.lower()
        matched_col = None

        if normalized_key in schema_dict:
            matched_col = normalized_key
        else:
            for c in df.columns:
                if c.lower() in normalized_key or normalized_key in c.lower():
                    matched_col = c
                    break

        if matched_col:
            df = df.withColumn(
                matched_col,
                coalesce(col(matched_col), col("rescued_map")[key])
            )
        else:
            df = df.withColumn(normalized_key, col("rescued_map")[key])

    return df.drop("rescued_map", "_rescued_data")


# =========================================
# 4. GENERATE TEMP SKEY (START 1001)
# =========================================
def generate_temp_skey(df, start_value=1001):

    window = Window.orderBy(monotonically_increasing_id())

    return df.withColumn(
        "temp_skey",
        row_number().over(window) + (start_value - 1)
    )


# =========================================
# 5. DATA QUALITY + DEDUP USING TEMP SKEY
# =========================================
def apply_data_quality(df, pk_col, trim_cols):

    for c in trim_cols:
        df = df.withColumn(c, trim(col(c)))

    df = df.select([col(c).alias(c.lower()) for c in df.columns])

    reject_df = df.filter(col(pk_col).isNull())
    df = df.filter(col(pk_col).isNotNull())

    df = df.fillna("unknown")

    window = Window.partitionBy("temp_skey").orderBy(current_timestamp().desc())

    df = df.withColumn("rn", row_number().over(window)) \
           .filter(col("rn") == 1) \
           .drop("rn")

    return df, reject_df


# =========================================
# 6. CREATE TABLE IF NOT EXISTS
# =========================================
def create_table_if_not_exists(df, table_name, sk_col):

    if not spark.catalog.tableExists(table_name):

        df = df.withColumn("StartDate", current_timestamp()) \
               .withColumn("EndDate", lit(None).cast("timestamp")) \
               .withColumn("IsCurrent", lit(1)) \
               .withColumn("hash", lit(None).cast("string")) \
               .withColumn(sk_col, lit(None).cast("long"))

        df.limit(0).write.format("delta").saveAsTable(table_name)

        print(f"✅ Created table: {table_name}")


# =========================================
# 7. SCD TYPE 2 WITH FINAL SKEY
# =========================================
def apply_scd2(df, table_name, pk, sk_col):

    if isinstance(pk, str):
        pk = [pk]

    df = df.withColumn("StartDate", current_timestamp()) \
           .withColumn("EndDate", lit(None).cast("timestamp")) \
           .withColumn("IsCurrent", lit(1))

    compare_cols = [c for c in df.columns if c not in ["StartDate", "EndDate", "IsCurrent", sk_col]]

    df = df.withColumn("hash", sha2(concat_ws("||", *compare_cols), 256))

    create_table_if_not_exists(df, table_name, sk_col)

    delta_table = DeltaTable.forName(spark, table_name)
    target_df = delta_table.toDF()

    max_id = target_df.agg(max(sk_col)).collect()[0][0]
    max_id = max_id if max_id else 1000

    window_sk = Window.orderBy(*pk)

    df = df.withColumn(sk_col, row_number().over(window_sk) + max_id)

    merge_cond = " AND ".join([f"t.{c} = s.{c}" for c in pk]) + " AND t.IsCurrent = 1"

    # Expire old records
    delta_table.alias("t").merge(
        df.alias("s"),
        merge_cond
    ).whenMatchedUpdate(
        condition="t.hash <> s.hash",
        set={
            "EndDate": current_timestamp(),
            "IsCurrent": lit(0)
        }
    ).execute()

    # Insert new records
    delta_table.alias("t").merge(
        df.alias("s"),
        merge_cond
    ).whenNotMatchedInsertAll().execute()


# =========================================
# 8. MAIN PIPELINE
# =========================================

# SOURCE TABLES
bronze_customers = spark.table("`eus-sales-catalog`.`bronze-layer`.`customer`")
bronze_products = spark.table("`eus-sales-catalog`.`bronze-layer`.`products`")



# -------- PRODUCTS --------
product_df = handle_schema_drift(bronze_products)

product_df = generate_temp_skey(product_df, 1001)

product_clean, _ = apply_data_quality(
    product_df,
    pk_col="product_id",
    trim_cols=["product_id", "product_name"]
)

apply_scd2(
    df=product_clean,
    table_name="`eus-sales-catalog`.`silver-layer`.`products`",
    pk=["product_id"],
    sk_col="skey_product_id"
)


# -------- CUSTOMERS --------

customers_df = bronze_customers.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name"))).withColumn("email", lower(col("email"))).withColumn("email", regexp_replace(col("email"), "@gmail.com", "@yahoo.com")).drop(col("first_name"), col("last_name")).filter(col("full_name").isNotNull())

customers_df = handle_schema_drift(customers_df)

customers_df = generate_temp_skey(customers_df, 1001)

customers_clean, _ = apply_data_quality(
    customers_df,
    pk_col="customer_id",
    trim_cols=["customer_id", "email", "full_name"]
)

apply_scd2(
    df=customers_clean,
    table_name="`eus-sales-catalog`.`silver-layer`.`customers`",
    pk=["customer_id"],
    sk_col="skey_customer_id"
)




In [0]:
# bronze_countries = spark.table("`adb-netflix`.`bronze-layer`.`countries`")
# bronze_category = spark.table("`adb-netflix`.`bronze-layer`.`category`")

# countries_df = handle_schema_drift(bronze_countries)
# silver_countries, _ = apply_data_quality(
#     countries_df, "show_id", ["country"]
# )

bronze_regions = spark.table("`eus-sales-catalog`.`bronze-layer`.`regions`")
bronze_store = spark.table("`eus-sales-catalog`.`bronze-layer`.`store`")

def apply_data_quality1(df, pk_col, trim_cols):

    for c in trim_cols:
        df = df.withColumn(c, trim(col(c)))

    df = df.select([col(c).alias(c.lower()) for c in df.columns])

    reject_df = df.filter(col(pk_col).isNull())
    df = df.filter(col(pk_col).isNotNull())

    df = df.fillna("unknown")

    df = df.dropDuplicates([pk_col])

    return df, reject_df


processed_regions = handle_schema_drift(bronze_regions)
bronze_regions = processed_regions if processed_regions is not None else bronze_regions

silver_regions, _ = apply_data_quality1(
   bronze_regions,
   pk_col = "region_id",
   trim_cols = ["region_id"]
)

silver_regions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`eus-sales-catalog`.`silver-layer`.`regions`")

processed_stores = handle_schema_drift(bronze_store)
bronze_stores = processed_stores if processed_stores is not None else bronze_store

silver_store, _ = apply_data_quality1(
   bronze_stores,
   pk_col = "store_id",
   trim_cols = ["store_id"]
)

silver_store.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`eus-sales-catalog`.`silver-layer`.`stores`")


print("✅ FULL PIPELINE COMPLETED SUCCESSFULLY")
